In [1]:
import os
import sys
import pandas as pd
import datetime as dt
from datetime import datetime
from dataretrieval import nwis
from dataretrieval import waterdata
import numpy as np
import requests
import io
from scripts import data

In [2]:
train1 = "10132000"
lat1 = 40.96772452
lon1 = -111.437699

train2 = "10136600"
lat2 = 41.13708333
lon2 = -111.9195556

train3 = "10137000"
lat3 = 41.223269
lon3 = -111.988117

test = "10136500" #test streamgage in the middle to avoid boundary effects 
lat_test = 41.1368878
lon_test = -111.8324384

train_ID = [train1,train2,train3]

start = "2016-01-01"
end="2025-12-31"
years= ",".join(str(y) for y in range(datetime.strptime(start, "%Y-%m-%d").year, #daymet didn't like dates for these gages, had to use years parameter!
                                       datetime.strptime(end, "%Y-%m-%d").year + 1))

Note that I reran my Homework 2 for each of the streamgages above to obtain the data files for SWE, please view the code in "https://github.com/eburgon2/Homework2" if you would like specific sytnax for how SWE values were obtained 

for my memory: 
train1 -> 330,392,393,763
train2 -> same, and 896,533,1145,1118,684,814
train3 -> same as 2
test -> same as 2



In [4]:
#only station 1 has less stations, we will include nan columns to fit the data to 10 in the LSTM
swe_stations_1 = ['330','392','393','763']
swe_stations_else = swe_stations_1 + ['533','684','814','896','1118','1145']

swe_1 = data.swe_set(swe_stations_1,train1)
swe_2 = data.swe_set(swe_stations_else,train2)
swe_3 = data.swe_set(swe_stations_else,train3)
swe_test = data.swe_set(swe_stations_else,test)

In [5]:
Q1 = data.discharge(train1,start,end)
Q2 = data.discharge(train2,start,end)
Q3 = data.discharge(train3,start,end)
Q_test = data.discharge(test,start,end)

In [6]:
daymet1 = data.normalize_daymet(data.daymet(train1,lat1,lon1,years))
daymet2 = data.normalize_daymet(data.daymet(train2,lat2,lon2,years))
daymet3 = data.normalize_daymet(data.daymet(train3,lat3,lon3,years))
daymet_Test = data.normalize_daymet(data.daymet(test,lat_test,lon_test,years))

The last data set we can consider is catchment characteristics. However, this isn't really something we can show over time, as they usually don't change drastically quickly (~3 years), so it likely won't be able to train the model well. I also looked through earth access options, and they mostly only give annual data. For this reason, we will be considering only the parameters we have shown above to avoid mistraining the model. 

In [7]:
#now masking the swe_1 to make sure it matches pytorch sizes
#note that the mask method will allow the LSTM to ignore NAN (aka won't break) 
swe_1 = swe_1.reindex(columns=list(swe_2.columns)) #creating Nans, we will mask on the datafram in the model portion of the assignment

training1 = data.full_data(daymet1,swe_1,train1)
training2 = data.full_data(daymet2,swe_2,train2)
training3 = data.full_data(daymet3,swe_3,train3)

testing = data.full_data(daymet_Test,swe_test,test)